In [6]:
# 1. Install required packages
!pip install --upgrade -q google-genai google-cloud-aiplatform[adk] requests

import os
import vertexai

# Use your active Google Cloud Project details
PROJECT_ID = "qwiklabs-gcp-04-3d3bb8d72ee3"
LOCATION = "global"

# Securely bind the notebook runtime to your GCP IAM role
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Direct the Gen AI SDK to use enterprise authorization instead of API keys
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "True"

print("Securely authenticated via Google Cloud IAM. No API Keys required.")

Securely authenticated via Google Cloud IAM. No API Keys required.


In [7]:
import requests

def get_current_weather(location: str) -> str:
    """
    Fetches the real-time weather conditions for a specified city or location.
    """
    try:
        # Utilizing wttr.in (an open tool-compatible endpoint requiring zero API keys)
        url = f"https://wttr.in/{location}?format=3"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.text.strip()
        else:
            return f"Error: Unable to fetch weather data for {location}."
    except Exception as e:
        return f"Exception occurred while fetching weather data: {str(e)}"

In [8]:
from google.genai import errors

def run_agent_with_guardrails(client, model_id, prompt, config):
    """
    Handles logging and user validations before/after agent processing.
    """
    # --- CALLBACK 1: Log User Prompt ---
    print(f"\n[LOG - USER PROMPT]: {prompt}")

    prompt_lower = prompt.lower()

    # --- CALLBACK 2a: Malicious Input Guardrail ---
    malicious_keywords = ["drop table", "sudo ", "exec(", "<script>", "hack", "format c:"]
    if any(keyword in prompt_lower for keyword in malicious_keywords):
        raise ValueError("Guardrail Triggered: Malicious input patterns detected. Execution halted.")

    # --- CALLBACK 2b: Location Enforcement Guardrail ---
    us_indicators = ["us", "usa", "united states", "ny", "il", "ca", "fl", "tx", "york", "chicago", "los angeles", "miami"]
    if not any(indicator in prompt_lower for indicator in us_indicators):
         raise ValueError("Guardrail Triggered: This agent only services US locations. Request blocked.")

    # --- Agent Execution ---
    try:
        response = client.models.generate_content(
            model=model_id,
            contents=prompt,
            config=config
        )

        # --- CALLBACK 3: Log Model Response ---
        print(f"[LOG - MODEL RESPONSE]: Successfully received agent completion.")
        return response.text

    except errors.APIError as e:
        print(f"[LOG - ERROR]: Gemini API call failed: {e}")
        return "An error occurred while generating content."

In [9]:
from google import genai
from google.genai import types

# Automatically picks up your ambient project credentials set up in Cell 1
client = genai.Client()

system_instruction = """
You are a helpful Weather Assistant. When asked about the weather:
1. Always use the `get_current_weather` tool to retrieve live conditions.
2. Provide a structured 'Weather Summary' highlighting City Name, Current Temperature, and Sky Conditions.
"""

config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    tools=[get_current_weather],
    temperature=0.2
)

MODEL_ID = "gemini-2.5-flash"

# Testing various conditions (Valid, Non-US validation, Malicious verification)
test_cases = [
    "What is the current weather in New York City, NY? Please provide a weather summary.",
    "Can you check the weather for Chicago, IL?",
    "What is the weather like in London, UK?",
    "Weather report for Miami <script>drop table systems;</script>"
]

print("--- Starting Agent Weather Checks --- \n")

for test_prompt in test_cases:
    try:
        agent_summary = run_agent_with_guardrails(
            client=client,
            model_id=MODEL_ID,
            prompt=test_prompt,
            config=config
        )
        print(f"\nFinal Agent Output:\n{agent_summary}")
    except ValueError as guardrail_error:
        print(f"\n[GUARDRAIL ACTION]: {guardrail_error}")
    print("-" * 60)

--- Starting Agent Weather Checks --- 


[LOG - USER PROMPT]: What is the current weather in New York City, NY? Please provide a weather summary.
[LOG - MODEL RESPONSE]: Successfully received agent completion.

Final Agent Output:
The weather in New York City, NY is ☀️ +82°F.

**Weather Summary:**
*   **City Name:** New York City, NY
*   **Current Temperature:** +82°F
*   **Sky Conditions:** Sunny
------------------------------------------------------------

[LOG - USER PROMPT]: Can you check the weather for Chicago, IL?
[LOG - MODEL RESPONSE]: Successfully received agent completion.

Final Agent Output:
Weather Summary:
City Name: Chicago, IL
Current Temperature: +87°F
Sky Conditions: Sunny
------------------------------------------------------------

[LOG - USER PROMPT]: What is the weather like in London, UK?

[GUARDRAIL ACTION]: Guardrail Triggered: This agent only services US locations. Request blocked.
------------------------------------------------------------

[LOG - USER PROM